In [ ]:
import optuna
import pandas as pd
from prophet import Prophet
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, MonthLocator, YearLocator
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import datetime
import holidays
import os
import logging
import sys
from prophet.diagnostics import cross_validation
from optuna.logging import set_verbosity
import json
from typing import Any, Dict, List, Tuple, Optional

In [ ]:
# downloading data

set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")
file_path_eacq = r'data_test.xlsx' 
file_path_eacq = r'data_eacq_SKB_fnal.xlsx' 
df = pd.read_excel(file_path_eacq)
df.head()

In [ ]:
# holidays 

today = datetime.date.today()
years = list(range(2022, today.year + 1))
country_holidays = holidays.Russia(years=years)
holidays_df = pd.DataFrame([
    {'ds': pd.to_datetime(dt), 'holiday': name, 'lower_window': 0, 'upper_window': 3}
    for dt, name in country_holidays.items()
])

In [ ]:
df_prophet_eacq = df.copy()  # замените на реальные имена столбцов
df_prophet_eacq.rename(columns={'VALUE_DAY': 'ds', 'FEE_RUR_AMT_SUM': 'y'}, inplace=True)
df_prophet_eacq.head()

In [ ]:
# calculation of loss functions (MAE, MAPE, etc)

def mape_loss(y_true, y_pred):
    """Mean Absolute Percentage Error"""
    return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9)))

def rmse_loss(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae_loss(y_true, y_pred):
    """Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def r2_loss(y_true, y_pred):
    """R2"""
    return r2_score(y_true, y_pred)

LOSS_FUNCS = {
    "mape": mape_loss,
    "rmse": rmse_loss,
    "mae": mae_loss,
    "r2": r2_loss
}


In [ ]:
# adding regressors for model 

def add_regressors(df: pd.DataFrame) -> pd.DataFrame:
    df['ds'] = pd.to_datetime(df['ds'])
    df['summer_season'] = df['ds'].dt.month.isin([6, 7, 8]).astype(int)
    return df

In [ ]:
# prophet from the box (wo Optuna)

model_old = Prophet(holidays=holidays_df,
            weekly_seasonality= True,
            yearly_seasonality= True,
            daily_seasonality= True,
            #changepoint_prior_scale= 0.05,
            interval_width= 0.95,)
model_old.fit(df_prophet_eacq)
future = pd.DataFrame({'ds': pd.date_range(start="2023-01-01", end="2026-12-31", freq='D')})
future.tail()
forecast_eacq_old = model_old.predict(future)

# visualization
fig1 = model_old.plot(forecast_eacq_old)
fig2 = model_old.plot_components(forecast_eacq_old)

# calculation of parameters for control measures

y_true = df_prophet_eacq['y'].values
y_pred = forecast_eacq_old['yhat'][:len(df_prophet_eacq)].values

r2 = r2_score(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred) * 100
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))

print(f"{'R²':<25} | {r2:^12.3f}")
print(f"{'MAE (thousands)':<25} | {mae/1000:^12,.0f}")
print(f"{'MAPE (%)':<25} | {mape:^12.1f}")
print(f"{'RMSE (thousands)':<25} | {rmse/1000:^12.1f}")

In [ ]:
# optimized Prophet with Optuna

# 0. turn off Optuna logging
optuna.logging.set_verbosity(optuna.logging.ERROR)
logging.basicConfig(level=logging.ERROR)
warnings.filterwarnings("ignore")
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
set_verbosity(optuna.logging.WARNING)


def run_optuna_search_revenue_cv(df_prophet_eacq, n_trials: int = 50, metric: str = {"mae", "mape","r2", "rmse"}) -> Dict:
    optuna.logging.set_verbosity(optuna.logging.ERROR)
    logging.basicConfig(level=logging.ERROR)
    warnings.filterwarnings("ignore")
    logging.getLogger('prophet').setLevel(logging.ERROR)
    logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
    set_verbosity(optuna.logging.WARNING)
    """Optimize Prophet hyperparameters using Optuna"""

    def objective(trial):
        # Suggest hyperparameters
        params = {
            "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.5, log=True),
            "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 10.0, log=True),
            "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 10.0, log=True),
            "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
            "n_changepoints": trial.suggest_int("n_changepoints", 10, 50),
            "changepoint_range": trial.suggest_float("changepoint_range", 0.8, 0.95),

        }

        df_train = add_regressors(df_prophet_eacq.copy())
        model = Prophet(
            holidays=holidays_df,
            weekly_seasonality=True,
            daily_seasonality=False,
            yearly_seasonality=False,
            interval_width=0.95,
            **params
        )
        # Optional seasonalities
        model.add_seasonality(name='yearly', period=365.25, fourier_order=3)
        model.add_seasonality(name='yearly_alt', period=365.25, fourier_order=15)

        # Fit model
        model.fit(df_train)

        # Cross-validation
        df_cv = cross_validation(model
                                 , initial="365 days"
                                 , period="31 days"
                                 , horizon="91 days"
                                 , parallel= None # "processes"
                                )
        y_true = df_cv["y"].values
        y_pred = df_cv["yhat"].values

        return LOSS_FUNCS[metric](y_true, y_pred)

    study = optuna.create_study(direction= "maximize")
    study.optimize(objective, n_trials=n_trials)
    
    return study.best_params


#  best_params from run_optuna_search_revenue_cv
best_params = run_optuna_search_revenue_cv(df_prophet_eacq, n_trials=50, metric= "r2")

# create a model with the best parameters
model = Prophet(
    holidays=holidays_df,
    weekly_seasonality=True,
    daily_seasonality=False,
    yearly_seasonality=False,
    interval_width=0.95,
    **best_params
)


model.add_seasonality(name='yearly', period=365.25, fourier_order=3)
model.add_seasonality(name='yearly_alt', period=365.25, fourier_order=15)


df_train = add_regressors(df_prophet_eacq.copy())
model.fit(df_train.rename(columns={"ds": "ds", "y": "y"}))

# create future dataframe
future = pd.DataFrame({'ds': pd.date_range(start="2023-01-01", end="2026-12-31", freq='D')})

# create forecast
forecast_eacq = model.predict(future)

# visualization
fig1 = model.plot(forecast_eacq)

y_true = df_prophet_eacq['y'].values
y_pred = forecast_eacq['yhat'][:len(df_prophet_eacq)].values

r2 = r2_score(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
mape = mean_absolute_percentage_error(y_true, y_pred) * 100
rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))

print(f"{'R²':<25} | {r2:^12.3f}")
print(f"{'MAE (thousands)':<25} | {mae/1000:^12,.0f}")
print(f"{'MAPE (%)':<25} | {mape:^12.1f}")
print(f"{'RMSE (thousands)':<25} | {rmse/1000:^12.1f}")

In [ ]:
# Merging tables
forecast_eacq['year_month'] = forecast_eacq['ds'].dt.to_period('M')
forecast_eacq_old['year_month'] = forecast_eacq_old['ds'].dt.to_period('M')
forecast_eacq_old.rename(columns={'yhat': 'y_old'}, inplace=True)

df_merged = forecast_eacq[['ds', 'year_month', 'yhat']].merge(
    forecast_eacq_old[['y_old','ds']],
    on='ds',
    how='left' 
)

df_merged.head()

In [ ]:
# Month aggregation
df_monthly = df_merged.groupby('year_month')[['yhat', 'y_old']].sum().reset_index()
df_monthly['year_month'] = df_monthly['year_month'].dt.to_timestamp() 
df_monthly = df_monthly[df_monthly['year_month'] > '2025-12']

# Creating plots
plt.figure(figsize=(18, 10))
plt.plot(df_monthly['year_month'], df_monthly['yhat'], label='prophet_x_optuna', marker='s', zorder=2)
plt.plot(df_monthly['year_month'], df_monthly['y_old'], label='prophet', marker='o', zorder=3)

plt.title('Forecast')
plt.xlabel('Month')
plt.ylabel('Amount')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

# adding values
for i, row in df_monthly.iterrows():
    x = row['year_month']
    # prophet_x_optuna (yhat) — сверху
    plt.annotate(f"{row['yhat']/1e9:.1f}", 
                 (x, row['yhat']), 
                 textcoords="offset points", 
                 xytext=(0, 10), 
                 ha='center', 
                 fontsize=9, 
                 zorder=5)
    # prophet (y_old) — снизу
    plt.annotate(f"{row['y_old']/1e9:.1f}", 
                 (x, row['y_old']), 
                 textcoords="offset points", 
                 xytext=(0, -15), 
                 ha='center', 
                 fontsize=9, 
                 zorder=5)

plt.tight_layout()
plt.show()